## EN3160 – Assignment 3: Neural Networks

**Name:** Deshan W.U.  
**Index Number:** 230130E

In [2]:
import torch
import torch .nn as nn
import torch .optim as optim
import torchvision
import torchvision .transforms as transforms
import matplotlib . pyplot as plt

Question 1

In [6]:
# 1. Data loading
transform = transforms.Compose(
    [transforms.ToTensor(),
     transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))])

batch_size = 50

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size,
                                          shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size,
                                         shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')



 36%|███▌      | 60.6M/170M [24:05<43:44, 41.9kB/s]  


KeyboardInterrupt: 

In [ ]:
# 2. Define Network Parameters
Din = 3 * 32 * 32  # Input size (flattened CIFAR-10 image size)
H = 100            # Number of nodes in the middle layer
K = 10             # Output size (number of classes in CIFAR-10)
std = 1e-2         # larger init std, needed for the hidden layer
 
# Initialize weights and biases
w1 = torch.randn(Din, H) * std       
b1 = torch.zeros(H)
w2 = torch.randn(H, K) * std * 10    
b2 = torch.zeros(K)
 
# Hyperparameters
iterations = 10    #  10 epochs
lr = 1e-1          # learning rate suited to cross-entropy
lr_decay = 0.9     # Learning rate decay
reg = 0            # Regularization
 
loss_history = []
 
# 3. Training Loop
for t in range(iterations):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # Get inputs and labels
        inputs, labels = data
        Ntr = inputs.shape[0]  # Batch size
        x_train = inputs.view(Ntr, -1)  # Flatten input to (Ntr, Din)
        y_train_onehot = nn.functional.one_hot(labels, K).float()  # Convert labels to one-hot
 
        # Forward pass
        h = torch.sigmoid(x_train.mm(w1) + b1)   # middle layer, sigmoid
        y_pred = h.mm(w2) + b2                   # Output layer
        # softmax for cross-entropy (max subtracted for stability)
        e = torch.exp(y_pred - y_pred.max(dim=1, keepdim=True).values)
        p = e / e.sum(dim=1, keepdim=True)
 
        # Loss calculation (Cross-entropy with regularization)
        loss = -(1 / Ntr) * torch.sum(y_train_onehot * torch.log(p + 1e-12)) \
               + reg * (torch.sum(w1 ** 2) + torch.sum(w2 ** 2))
        loss_history.append(loss.item())
        running_loss += loss.item()
 
        # Backpropagation
        dy_pred = (1.0 / Ntr) * (p - y_train_onehot)    # softmax + cross-entropy gradient
        dw2 = h.t().mm(dy_pred) + reg * w2
        db2 = dy_pred.sum(dim=0)
        dh = dy_pred.mm(w2.t())                         # back through middle layer
        dz1 = dh * h * (1 - h)                          # sigmoid derivative
        dw1 = x_train.t().mm(dz1) + reg * w1
        db1 = dz1.sum(dim=0)
 
        # Parameter update
        w1 -= lr * dw1
        b1 -= lr * db1
        w2 -= lr * dw2
        b2 -= lr * db2
 
    # Print loss for every epoch
    if t % 1 == 0:
        print(f"Epoch {t + 1}/{iterations}, Loss: {running_loss / len(trainloader)}")
 
    # Learning rate decay
    lr *= lr_decay
 
# 4. Plotting the Loss History
plt.plot(loss_history)
plt.title("Loss History")
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.show()
 
# 5. Calculate Accuracy on Training Set
correct_train = 0
total_train = 0
with torch.no_grad():
    for data in trainloader:
        inputs, labels = data
        Ntr = inputs.shape[0]
        x_train = inputs.view(Ntr, -1)
        y_train_onehot = nn.functional.one_hot(labels, K).float()
 
        # Forward pass 
        h = torch.sigmoid(x_train.mm(w1) + b1)
        y_train_pred = h.mm(w2) + b2
 
        predicted_train = torch.argmax(y_train_pred, dim=1)
        total_train += labels.size(0)
        correct_train += (predicted_train == labels).sum().item()
 
train_acc = 100 * correct_train / total_train
print(f"Training accuracy: {train_acc:.2f}%")
 
# 6. Calculate Accuracy on Test Set
correct_test = 0
total_test = 0
with torch.no_grad():
    for data in testloader:
        inputs, labels = data
        Nte = inputs.shape[0]
        x_test = inputs.view(Nte, -1)
        y_test_onehot = nn.functional.one_hot(labels, K).float()
 
        # Forward pass 
        h = torch.sigmoid(x_test.mm(w1) + b1)
        y_test_pred = h.mm(w2) + b2
 
        predicted_test = torch.argmax(y_test_pred, dim=1)
        total_test += labels.size(0)
        correct_test += (predicted_test == labels).sum().item()
 
test_acc = 100 * correct_test / total_test
print(f"Test accuracy: {test_acc:.2f}%")
 